# Day 05 · Tool 기반 MCP 기능 구현
FastMCP로 행동하는 도구(Tool)를 제대로 만듭니다 — 부작용, 입력 검증과 에러, 구조화된 출력과 annotations, Context와 진행 보고, 그리고 검색을 도구로 노출하기.
Day03~04처럼 별도 프로세스 없이 in-process Client로 바로 검증합니다.

## Lab 0 · 셋업

In [ ]:
!pip install -q fastmcp

In [ ]:
import fastmcp
print("fastmcp", fastmcp.__version__)

from fastmcp import FastMCP, Client, Context
mcp = FastMCP("Day05ToolServer")

## Lab 1 · 부작용 도구
Resource(읽기 전용, Day04)와 달리 Tool은 상태를 바꾸는 행동을 한다. 여기서는 할 일 목록에 저장하고 완료 처리한다.

In [ ]:
TASKS = []

@mcp.tool
def add_task(title: str, priority: str = "보통") -> dict:
    """할 일을 추가한다 (부작용: 목록에 저장)"""
    task = {"id": len(TASKS) + 1, "title": title, "priority": priority, "done": False}
    TASKS.append(task)
    return {"저장됨": task, "총개수": len(TASKS)}

@mcp.tool
def complete_task(task_id: int) -> str:
    """할 일을 완료 처리한다"""
    for t in TASKS:
        if t["id"] == task_id:
            t["done"] = True
            return f"완료: {t['title']}"
    return f"id {task_id} 인 할 일이 없습니다."

async with Client(mcp) as c:
    print((await c.call_tool("add_task", {"title": "슬라이드 검토", "priority": "높음"})).data)
    print((await c.call_tool("add_task", {"title": "노트북 검증"})).data)
    print("완료 처리:", (await c.call_tool("complete_task", {"task_id": 1})).data)

## Lab 2 · 입력 검증과 친절한 에러
잘못된 인자에는 무엇이 왜 틀렸는지 알려주는 에러를 던진다. 클라이언트는 그 메시지를 받는다.

In [ ]:
ALLOWED = {"낮음", "보통", "높음"}

@mcp.tool
def set_priority(task_id: int, priority: str) -> str:
    """우선순위를 바꾼다. 허용값이 아니면 친절한 에러를 던진다."""
    if priority not in ALLOWED:
        raise ValueError(f"priority는 {sorted(ALLOWED)} 중 하나여야 합니다. 받은 값: '{priority}'")
    for t in TASKS:
        if t["id"] == task_id:
            t["priority"] = priority
            return f"{t['title']} 우선순위 → {priority}"
    raise ValueError(f"id {task_id} 인 할 일이 없습니다.")

async with Client(mcp) as c:
    print((await c.call_tool("set_priority", {"task_id": 2, "priority": "낮음"})).data)
    try:
        await c.call_tool("set_priority", {"task_id": 2, "priority": "긴급"})
    except Exception as e:
        print("에러 전달됨:", str(e).splitlines()[-1])

## Lab 3 · 구조화된 출력과 annotations
dict 대신 스키마가 분명한 구조(Pydantic)로 반환하고, 이 도구가 읽기 전용인지 등을 annotations로 알린다.

In [ ]:
from pydantic import BaseModel

class TaskStats(BaseModel):
    total: int
    done: int
    pending: int

@mcp.tool(annotations={"readOnlyHint": True})
def task_stats() -> TaskStats:
    """할 일 통계를 구조화해 반환한다 (읽기 전용)"""
    done = sum(t["done"] for t in TASKS)
    return TaskStats(total=len(TASKS), done=done, pending=len(TASKS) - done)

async with Client(mcp) as c:
    anns = {t.name: t.annotations for t in await c.list_tools()}
    print("task_stats 읽기전용?:", anns["task_stats"].readOnlyHint)
    r = await c.call_tool("task_stats", {})
    print("구조화 출력:", r.data, "| total =", r.data.total)

## Lab 4 · Context와 진행 보고
오래 걸리는 작업은 Context(ctx)로 로그와 진행률을 보고한다. 인자에 `ctx: Context`를 넣으면 자동 주입된다.
클라이언트는 `progress_handler`로 진행 상황을 받는다.

In [ ]:
import asyncio

@mcp.tool
async def bulk_import(count: int, ctx: Context) -> dict:
    """여러 건을 가져오며 진행률을 보고한다"""
    await ctx.info(f"{count}건 가져오기 시작")
    for i in range(count):
        TASKS.append({"id": len(TASKS) + 1, "title": f"가져온 항목 {i+1}", "priority": "보통", "done": False})
        await ctx.report_progress(i + 1, count)
        await asyncio.sleep(0)          # 실제로는 파일/DB/네트워크 I/O
    await ctx.info("완료")
    return {"가져옴": count, "총개수": len(TASKS)}

async def on_progress(progress, total, message=None):
    print(f"  진행 {progress}/{total}")

async with Client(mcp, progress_handler=on_progress) as c:
    print((await c.call_tool("bulk_import", {"count": 3})).data)

## Lab 5 · 검색을 도구로
Day01~02의 검색을 MCP Tool로 노출하면, 에이전트가 이 도구를 호출해 근거를 찾는다.
여기서는 형태를 익히기 위해 간단한 키워드 검색을 쓴다. 실제로는 Day02의 `advanced_rag`(하이브리드+리랭커)를 이 자리에 넣는다.

In [ ]:
DOCS = [
    "연차 휴가는 사용 3영업일 전까지 신청하고 팀장 승인을 받는다.",
    "비용 정산은 영수증 첨부 후 재무팀이 최종 승인한다.",
    "재택근무는 주 2회까지 가능하며 전날 18시까지 공유한다.",
    "에러코드 ERR-4021은 인증 토큰 만료를 의미하며 재로그인으로 해결한다.",
]

@mcp.tool(annotations={"readOnlyHint": True})
def search_docs(query: str, k: int = 2) -> list:
    """사내 문서에서 질문과 겹치는 단어가 많은 문장을 찾는다 (에이전트의 검색 도구)"""
    ranked = sorted(DOCS, key=lambda d: -sum(w in d for w in query.split()))
    return ranked[:k]

async with Client(mcp) as c:
    r = await c.call_tool("search_docs", {"query": "연차 신청 며칠 전", "k": 2})
    print("검색 결과:")
    for line in r.data:
        print(" -", line)

## Lab 6 · 도전 (선택)
- **A** 이 서버를 `server.py`로 저장하고 `mcp.run()`을 붙여 터미널에서 `fastmcp dev`로 브라우저 검증.
- **B** 삭제처럼 되돌리기 힘든 도구에 `annotations={"destructiveHint": True}`를 달고, 삭제 전 확인 단계를 넣기.
- **C** `search_docs`를 Day02의 `advanced_rag`(하이브리드+리랭커)로 교체해 진짜 검색 도구로.
- **D** Claude Desktop 등 실제 MCP 클라이언트에 이 서버를 연결.

```python
# server.py 로 저장해 실행하는 형태
if __name__ == "__main__":
    mcp.run()   # 기본 stdio
```

## 산출물 체크리스트
- [ ] 부작용 도구(저장/완료)와 읽기 전용 도구를 구분해 만들었다
- [ ] 잘못된 입력에 친절한 에러를 던진다
- [ ] 구조화된 출력(Pydantic)과 annotations(읽기 전용 등)를 붙였다
- [ ] Context로 진행률을 보고하고 클라이언트에서 받았다
- [ ] 검색을 Tool로 노출해 에이전트가 쓸 수 있게 했다